# Vector Database

### Setup Notebook

#### Set System Path (Required for imports)

In [1]:

from pathlib import Path
import sys

def get_project_root():
    project_root = Path.cwd().resolve()
    while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
        project_root = project_root.parent
    return project_root

def setup_notebook():
    project_root = get_project_root()

    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))

setup_notebook()

#### Import Basic Libraries

In [2]:
from langchain_chroma import Chroma

In [3]:
from src.config import settings
from src.config.loader import load_config
from src.utils.logger import Logger
from src.utils import helper

Project root: F:\Github\Personal\real-world-rag-system


In [4]:
from src.rag_components.embeddings import EmbedderFactory

In [5]:
env_settings = None
config = None
logger = None

In [6]:
# settings.logging.log_level = settings.logging.log_level.split("#")[0].strip().upper()

logger = Logger()  # Initialize logger before loading environment settings
env_settings = settings


Logger directory: F:\Github\Personal\real-world-rag-system\logs
2026-06-21 10:16:37,286 | INFO	| real_world_rag | RAG System started


#### Load Config

In [7]:

def load_config_from_file(config_path: str = None):
    """
    Load configuration from file and set global config variable        
    """

    global config

    if config_path is None:
        config_path = "src/rag_flow_config/baseline.yaml"
    
    logger.info(f"Loading configuration from file... {config_path}")

    config = load_config(config_path)
    logger.debug(f"Data loader config: {config.data_loader}")


load_config_from_file(settings.pipeline_config_file)

2026-06-21 10:16:37,306 | INFO	| real_world_rag | Loading configuration from file... src/rag_flow_config/rag_bench_data.yaml
2026-06-21 10:16:37,311 | DEBUG	| real_world_rag | Data loader config: source=<DataLoaderSource.HUGGINGFACE: 'huggingface'> path=None dataset_name='suniltvl/ragbench' subset='emanual' split='test' cache_dir='./data/hf_cache' data_dir=None streaming=False file_extension='' base_url=None


## Create Vector Store

In [8]:
# from src.rag_components.vector_stores.factory import VectorStoreFactory

# def create_vector_store():
#     vector_store = VectorStoreFactory.create(
#         provider="chroma",
#         collection_name="baseline"
#     )
#     return vector_store


In [9]:

def get_embedder():
    """
    Embedding component of the RAG system. This function initializes the embedder based on the configuration and embeds the data.
    """
    logger.info("embedding initialized")
    embedder = EmbedderFactory.create(config.embedding)

    return embedder


In [10]:
# from langchain_huggingface import HuggingFaceEmbeddings

# print("before")

# emb = HuggingFaceEmbeddings(
#     model_name="BAAI/bge-small-en-v1.5"
# )

# print("after")

In [11]:
# from sentence_transformers import SentenceTransformer
# print("imported")

In [12]:
# import sentence_transformers
# print("ok")

In [13]:
# import torch

# print(torch.__version__)
# print(torch.cuda.is_available())
# print(torch.cuda.device_count())

In [14]:
# from sentence_transformers import SentenceTransformer

# print("before")

# model = SentenceTransformer(
#     "BAAI/bge-small-en-v1.5",
#     device="cpu"
# )

# print("after")

In [15]:
# from sentence_transformers import SentenceTransformer


# model = SentenceTransformer(
#     "BAAI/bge-small-en-v1.5",
#     device="cuda"
# )

# print("cuda loaded")

In [16]:

# embedder = EmbedderFactory.create(config.embedding)
# print(type(embedder))

In [17]:
def get_vector_client():
    db_path_relative = "db/chroma_db"
    is_db_path_exist = helper.is_dir_in_project(db_path_relative)

    if not is_db_path_exist:
        helper.create_dir(db_path_relative)
        
    db_path = helper.get_dir_in_project(db_path_relative)

    chroma_client = Chroma(
                collection_name="baseline",
                persist_directory=db_path,
                embedding_function=get_embedder()
            )
            
    return chroma_client



In [18]:
ch_cl = get_vector_client()


collection = ch_cl._client.get_or_create_collection(
    name="baseline"
)

documents = [
    "LangChain is a framework for LLM applications",
    "ChromaDB is an open-source vector database",
    "RAG stands for Retrieval Augmented Generation",
    "FAISS is developed by Facebook AI Research",
    "BGE is a popular embedding model"
]

ids = [f"doc_{i}" for i in range(len(documents))]

collection.add(
    ids=ids,
    documents=documents
)

py_table = '''
 [["Name", "Age", "Title"], ["Leigh R Fox", "47", "President and Chief Executive Officer"], ["Andrew R Kaiser", "51", "Chief Financial Officer"], ["Christi H. Cornette", "64", "Chief Culture Officer"], ["Thomas E. Simpson", "47", "Chief Operating Officer"], ["Christopher J. Wilson", "54", "Vice President and General Counsel"], ["Joshua T. Duckworth", "41", "Vice President of Treasury, Corporate Finance and Investor Relations"], ["Suzanne E. Maratta", "37", "Vice President and Corporate Controller"]]
 '''

collection.add(
    ids=["doc_5"],
    documents=[py_table]
)

print(collection.count())


2026-06-21 10:16:37,486 | INFO	| real_world_rag | embedding initialized
7


In [19]:
py_table[1]

collection = ch_cl._client.get_or_create_collection(
    name="baseline"
)

## Add
collection.add(
    ids=["doc_5"],
    documents=["ABC DEF BGE is a popular embedding"]
)

## Update
collection.update(
    ids=["doc_5"],
    documents=["sdfsdfsdf fdsf sdfdsf dsf dsf dsfd  "]
)

## Upsert
collection.upsert(
    ids=["doc_5"],
    documents=["sdfsdfsdf fdsf sdfdsf dsf dsf dsfd Updated  "]
)

## Update Metadata
collection.update(
    ids=["doc_5"],
    metadatas=[{"source": "updated", "dataset_name": "tatqa", "model": "bge"}]
)


# collection.count()

In [ ]:

## Query text with collection
results = collection.query(
    query_texts=["ABC DEF BGE is a popular embedding"],
    n_results=10
)


# ## Get all documents
# results = collection.get()


# ## Get by id
# results = collection.get(
#     ids=["doc_5"]
# )


# ## Filter with metadata
# results = collection.get(
#     where={"dataset_name": "tatqa"}
# )

# ## Filter with multiple conditions
# results = collection.get(
#     where={
#         "dataset_name": "tatqa",
#         "generation_model_name": "gpt-3.5-turbo"
#     }
# )

results

{'ids': [['doc_6', 'doc_4', 'doc_1', 'doc_3', 'doc_0', 'doc_2', 'doc_5']],
 'embeddings': None,
 'documents': [['ABC DEF BGE is a popular embedding model ',
   'BGE is a popular embedding model',
   'ChromaDB is an open-source vector database',
   'FAISS is developed by Facebook AI Research',
   'LangChain is a framework for LLM applications',
   'RAG stands for Retrieval Augmented Generation',
   'sdfsdfsdf fdsf sdfdsf dsf dsf dsfd Updated  ']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None,
   None,
   None,
   None,
   None,
   None,
   {'model': 'bge', 'source': 'updated', 'dataset_name': 'tatqa'}]],
 'distances': [[0.13505694270133972,
   0.4632197618484497,
   1.4818013906478882,
   1.524207592010498,
   1.6821335554122925,
   1.6904494762420654,
   1.7668708562850952]]}

In [ ]:

print("Collection count:", ch_cl._collection.count())

In [22]:
query = "what is age of Leigh?"

results = collection.query(
    query_texts=[query],
    n_results=3
)

(results)

{'ids': [['doc_3', 'doc_5', 'doc_6']],
 'embeddings': None,
 'documents': [['FAISS is developed by Facebook AI Research',
   'sdfsdfsdf fdsf sdfdsf dsf dsf dsfd Updated  ',
   'ABC DEF BGE is a popular embedding model ']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[None,
   {'source': 'updated', 'dataset_name': 'tatqa', 'model': 'bge'},
   None]],
 'distances': [[1.7436352968215942, 1.7820954322814941, 1.818414330482483]]}

In [ ]:
query_vector = get_embedder().embed_query("open-source")


collection.query(
    query_embeddings=[query_vector],
    # where={"domain":"finance"}
)